# NBA Data Repository Builder

Builds a **local cache of NBA data** from `stats.nba.com` (via `nba_api`), so analysis and modeling never depend on live API calls or rate limits.

| Repository | Endpoint | Grain | Calls needed |
|---|---|---|---|
| Active players | `CommonAllPlayers` | one row per active player | 1 |
| Team games | `LeagueGameLog` (`T`) | one row per team per game | 1 per season per season type |
| Player game logs | `LeagueGameLog` (`P`) | one row per player per game | 1 per season per season type |
| Player season stats | `LeagueDashPlayerStats` | one row per player per season | 1 per season per season type |
| Play-by-play | `PlayByPlayV3` | one row per game event | **1 per game** (~1,230 per season) |

**How caching works**
- Every API response is saved to `data/raw/` as a Parquet file. Re-running a cell loads from disk instead of calling the API.
- Completed seasons are fetched **once**. The current season re-fetches on every run so new games get picked up.
- Play-by-play is cached **per game** and is resumable: if it stops halfway, re-run the cell and it continues where it left off.
- Combined tables are written to `data/processed/`.

> Run this on your own computer. `stats.nba.com` often blocks cloud servers (Colab, AWS, and similar).

In [ ]:
# Run once to install dependencies
# %pip install nba_api pandas pyarrow tqdm

## 1. Setup

In [ ]:
import random
import time
from datetime import date
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from nba_api.stats.endpoints import (
    commonallplayers,
    leaguedashplayerstats,
    leaguegamelog,
    playbyplayv3,
)
from nba_api.stats.static import players as static_players

pd.set_option("display.max_columns", 50)

In [ ]:
# ---- Configuration ----
DATA_DIR = Path("data")
RAW = DATA_DIR / "raw"
PROCESSED = DATA_DIR / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

def current_nba_season(today=None):
    """NBA seasons start in October: Nov 2026 -> '2026-27', Mar 2026 -> '2025-26'."""
    today = today or date.today()
    start = today.year if today.month >= 10 else today.year - 1
    return f"{start}-{str(start + 1)[-2:]}"

CURRENT_SEASON = current_nba_season()
SEASONS = ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26', '2026-27']   # one cell per season below
SEASON_TYPES = ["Regular Season", "Playoffs"]

REQUEST_PAUSE = 0.8   # seconds between API calls (be polite; raise this if you get blocked)
TIMEOUT = 60          # seconds per request
MAX_RETRIES = 5
BACKOFF_BASE = 5      # seconds; doubles each retry: 5, 10, 20, 40...

print("Current season:", CURRENT_SEASON)

In [ ]:
# ---- Helpers: retry + cache ----
def slug(text):
    return text.lower().replace(" ", "_")

def fetch_with_retry(request_fn, label, retries=MAX_RETRIES):
    """Call the API with exponential backoff. Pauses after every successful call."""
    for attempt in range(1, retries + 1):
        try:
            result = request_fn()
            time.sleep(REQUEST_PAUSE)
            return result
        except Exception as e:
            if attempt == retries:
                raise
            wait = BACKOFF_BASE * 2 ** (attempt - 1) + random.uniform(0, 1)
            print(f"  {label}: attempt {attempt} failed ({type(e).__name__}), retrying in {wait:.0f}s")
            time.sleep(wait)

def load_or_fetch(path, request_fn, label, refresh=False):
    """Load a cached Parquet file, or fetch from the API and cache it."""
    path = Path(path)
    if path.exists() and not refresh:
        df = pd.read_parquet(path)
        print(f"{label}: {len(df):,} rows (cache)")
        return df
    df = fetch_with_retry(request_fn, label)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    print(f"{label}: {len(df):,} rows (fetched -> {path})")
    return df

def should_refresh(season, refresh):
    """Default: only the in-progress season is re-fetched."""
    return (season == CURRENT_SEASON) if refresh is None else refresh

def union(frames, name):
    """Combine per-season tables, warn about missing seasons, save to data/processed."""
    missing = [s for s in SEASONS if s not in frames]
    if missing:
        print(f"Warning: {name} is missing seasons {missing} (run those cells first)")
    df = pd.concat([frames[s] for s in SEASONS if s in frames], ignore_index=True)
    df.to_parquet(PROCESSED / f"{name}.parquet", index=False)
    print(f"{name}: {len(df):,} rows -> {PROCESSED / f'{name}.parquet'}")
    return df

## 2. Active players
One call. Includes current team. Set `refresh=True` to update after trades or signings.

In [ ]:
active_players_raw = load_or_fetch(
    RAW / "players" / f"all_players_{CURRENT_SEASON}.parquet",
    lambda: commonallplayers.CommonAllPlayers(
        is_only_current_season=1, season=CURRENT_SEASON, timeout=TIMEOUT
    ).get_data_frames()[0],
    "Active players",
    refresh=False,
)
active_players = active_players_raw[active_players_raw["ROSTERSTATUS"] == 1].copy()
active_players.to_parquet(PROCESSED / "active_players.parquet", index=False)
print(f"{len(active_players):,} active players")
active_players.head()

In [ ]:
# Offline backup: nba_api ships a static player list (no API call, but no team info)
static_active = pd.DataFrame(static_players.get_active_players())
print(f"{len(static_active):,} players in static list")
static_active.head()

## 3. Team games (one row per team per game)
`LeagueGameLog` with `T` returns each game twice, once per team. Section 3b turns it into one row per game with home/away and the final result, which becomes the **label** for the win probability model.

In [ ]:
def fetch_team_games(season, refresh=None):
    frames = []
    for st in SEASON_TYPES:
        df = load_or_fetch(
            RAW / "team_games" / season / f"{slug(st)}.parquet",
            lambda: leaguegamelog.LeagueGameLog(
                season=season, season_type_all_star=st,
                player_or_team_abbreviation="T", timeout=TIMEOUT,
            ).get_data_frames()[0],
            f"Team games {season} {st}",
            refresh=should_refresh(season, refresh),
        )
        frames.append(df.assign(SEASON=season, SEASON_TYPE=st))
    return pd.concat(frames, ignore_index=True)

team_games = {}   # season -> DataFrame

In [ ]:
team_games["2021-22"] = fetch_team_games("2021-22")

In [ ]:
team_games["2022-23"] = fetch_team_games("2022-23")

In [ ]:
team_games["2023-24"] = fetch_team_games("2023-24")

In [ ]:
team_games["2024-25"] = fetch_team_games("2024-25")

In [ ]:
team_games["2025-26"] = fetch_team_games("2025-26")

In [ ]:
team_games["2026-27"] = fetch_team_games("2026-27")

### 3a. Union all seasons

In [ ]:
all_team_games = union(team_games, "team_games")
all_team_games.groupby(["SEASON", "SEASON_TYPE"]).size().unstack()

### 3b. One row per game (home vs. away)

In [ ]:
keep = ["GAME_ID", "GAME_DATE", "SEASON", "SEASON_TYPE", "TEAM_ID", "TEAM_ABBREVIATION", "PTS"]
home = all_team_games.loc[all_team_games["MATCHUP"].str.contains(" vs. "), keep]
away = all_team_games.loc[all_team_games["MATCHUP"].str.contains(" @ "), ["GAME_ID", "TEAM_ID", "TEAM_ABBREVIATION", "PTS"]]

games = (
    home.rename(columns={"TEAM_ID": "HOME_TEAM_ID", "TEAM_ABBREVIATION": "HOME_TEAM", "PTS": "HOME_PTS"})
    .merge(
        away.rename(columns={"TEAM_ID": "AWAY_TEAM_ID", "TEAM_ABBREVIATION": "AWAY_TEAM", "PTS": "AWAY_PTS"}),
        on="GAME_ID", how="inner", validate="one_to_one",
    )
)
games["HOME_WIN"] = (games["HOME_PTS"] > games["AWAY_PTS"]).astype(int)
games = games.sort_values(["GAME_DATE", "GAME_ID"]).reset_index(drop=True)

expected = all_team_games["GAME_ID"].nunique()
assert len(games) == expected, f"{expected - len(games)} games missing a home or away row"
games.to_parquet(PROCESSED / "games.parquet", index=False)
print(f"{len(games):,} games | home win rate {games['HOME_WIN'].mean():.1%}")
games.tail()

## 4. Player game logs (one row per player per game)
Same `LeagueGameLog` endpoint with `P`: a full season of box scores in a single call.

In [ ]:
def fetch_player_game_logs(season, refresh=None):
    frames = []
    for st in SEASON_TYPES:
        df = load_or_fetch(
            RAW / "player_game_logs" / season / f"{slug(st)}.parquet",
            lambda: leaguegamelog.LeagueGameLog(
                season=season, season_type_all_star=st,
                player_or_team_abbreviation="P", timeout=TIMEOUT,
            ).get_data_frames()[0],
            f"Player game logs {season} {st}",
            refresh=should_refresh(season, refresh),
        )
        frames.append(df.assign(SEASON=season, SEASON_TYPE=st))
    return pd.concat(frames, ignore_index=True)

player_game_logs = {}

In [ ]:
player_game_logs["2021-22"] = fetch_player_game_logs("2021-22")

In [ ]:
player_game_logs["2022-23"] = fetch_player_game_logs("2022-23")

In [ ]:
player_game_logs["2023-24"] = fetch_player_game_logs("2023-24")

In [ ]:
player_game_logs["2024-25"] = fetch_player_game_logs("2024-25")

In [ ]:
player_game_logs["2025-26"] = fetch_player_game_logs("2025-26")

In [ ]:
player_game_logs["2026-27"] = fetch_player_game_logs("2026-27")

### 4a. Union all seasons

In [ ]:
all_player_game_logs = union(player_game_logs, "player_game_logs")
all_player_game_logs.groupby(["SEASON", "SEASON_TYPE"]).size().unstack()

## 5. Player season stats (one row per player per season)
`LeagueDashPlayerStats`: season averages. Change `PER_MODE` to `"Totals"` or `"Per36"` if needed, or `MEASURE` to `"Advanced"` for usage, true shooting, net rating, and similar.

In [ ]:
PER_MODE = "PerGame"
MEASURE = "Base"

def fetch_player_season_stats(season, refresh=None):
    frames = []
    for st in SEASON_TYPES:
        df = load_or_fetch(
            RAW / "player_season_stats" / f"{slug(MEASURE)}_{slug(PER_MODE)}" / season / f"{slug(st)}.parquet",
            lambda: leaguedashplayerstats.LeagueDashPlayerStats(
                season=season, season_type_all_star=st,
                per_mode_detailed=PER_MODE, measure_type_detailed_defense=MEASURE,
                timeout=TIMEOUT,
            ).get_data_frames()[0],
            f"Player season stats {season} {st}",
            refresh=should_refresh(season, refresh),
        )
        frames.append(df.assign(SEASON=season, SEASON_TYPE=st))
    return pd.concat(frames, ignore_index=True)

player_season_stats = {}

In [ ]:
player_season_stats["2021-22"] = fetch_player_season_stats("2021-22")

In [ ]:
player_season_stats["2022-23"] = fetch_player_season_stats("2022-23")

In [ ]:
player_season_stats["2023-24"] = fetch_player_season_stats("2023-24")

In [ ]:
player_season_stats["2024-25"] = fetch_player_season_stats("2024-25")

In [ ]:
player_season_stats["2025-26"] = fetch_player_season_stats("2025-26")

In [ ]:
player_season_stats["2026-27"] = fetch_player_season_stats("2026-27")

### 5a. Union all seasons

In [ ]:
all_player_season_stats = union(player_season_stats, "player_season_stats")
all_player_season_stats.groupby(["SEASON", "SEASON_TYPE"]).size().unstack()

## 6. Play-by-play (one call per game)
This is the input for the win probability model, and the slow part: ~1,230 regular-season games per season, roughly **20–40 minutes per season**.

- Each game is saved as its own file in `data/raw/pbp/<season>/`, so it's **resumable**. Interrupt any time and re-run to continue.
- Already-cached games are skipped, so for the current season only new games are fetched.
- **First run:** set `MAX_GAMES = 20` to test, then set it back to `None`.

In [ ]:
MAX_GAMES = None   # e.g. 20 for a quick test run

def game_ids_for(season):
    return games.loc[games["SEASON"] == season, "GAME_ID"].tolist()

def fetch_season_pbp(season, max_games=MAX_GAMES):
    folder = RAW / "pbp" / season
    folder.mkdir(parents=True, exist_ok=True)
    ids = game_ids_for(season)
    todo = [g for g in ids if not (folder / f"{g}.parquet").exists()]
    print(f"{season}: {len(ids):,} games | {len(ids) - len(todo):,} cached | {len(todo):,} to fetch")
    if max_games:
        todo = todo[:max_games]

    failed = []
    for gid in tqdm(todo, desc=season):
        try:
            df = fetch_with_retry(
                lambda: playbyplayv3.PlayByPlayV3(game_id=gid, timeout=TIMEOUT).get_data_frames()[0],
                f"PBP {gid}",
            )
            df.to_parquet(folder / f"{gid}.parquet", index=False)
        except Exception as e:
            failed.append(gid)
            print(f"  gave up on {gid}: {type(e).__name__}")
    if failed:
        print(f"{len(failed)} failed (re-run this cell to retry them): {failed[:10]}")
    return failed

def load_season_pbp(season):
    files = sorted((RAW / "pbp" / season).glob("*.parquet"))
    if not files:
        return pd.DataFrame()
    df = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)
    return df.assign(SEASON=season)

pbp_failed = {}

In [ ]:
pbp_failed["2021-22"] = fetch_season_pbp("2021-22")

In [ ]:
pbp_failed["2022-23"] = fetch_season_pbp("2022-23")

In [ ]:
pbp_failed["2023-24"] = fetch_season_pbp("2023-24")

In [ ]:
pbp_failed["2024-25"] = fetch_season_pbp("2024-25")

In [ ]:
pbp_failed["2025-26"] = fetch_season_pbp("2025-26")

In [ ]:
pbp_failed["2026-27"] = fetch_season_pbp("2026-27")

### 6a. Union all seasons

In [ ]:
pbp_frames = {s: load_season_pbp(s) for s in SEASONS}
pbp_frames = {s: df for s, df in pbp_frames.items() if not df.empty}
all_pbp = union(pbp_frames, "play_by_play")
print(f"{all_pbp['gameId'].nunique():,} games with play-by-play")
all_pbp.head()

## 7. Repository summary

In [ ]:
coverage = pd.DataFrame({
    "games": games.groupby("SEASON").size(),
    "games_with_pbp": pd.Series({s: len(list((RAW / "pbp" / s).glob("*.parquet"))) for s in SEASONS}),
    "player_game_rows": all_player_game_logs.groupby("SEASON").size(),
    "players_with_stats": all_player_season_stats.groupby("SEASON")["PLAYER_ID"].nunique(),
}).fillna(0).astype(int)
coverage["pbp_coverage"] = (coverage["games_with_pbp"] / coverage["games"].where(coverage["games"] > 0)).map("{:.0%}".format)
print("Active players:", len(active_players))
coverage

In [ ]:
# Disk usage of the local repository
for folder in sorted(p for p in DATA_DIR.glob("*/*") if p.is_dir()):
    size = sum(f.stat().st_size for f in folder.rglob("*.parquet"))
    print(f"{str(folder):<35} {size / 1e6:8.1f} MB")